In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_percentage_error, root_mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
import lightgbm as lgb
import xgboost as xgb
import optuna
import mlflow
import mlflow.xgboost
import mlflow.lightgbm
import mlflow.sklearn
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
mlflow.set_experiment("warsaw-apartment-price")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")
print("MLflow ready. Run `mlflow ui` in the project root to browse runs.")

In [ ]:
df = pd.read_parquet('../data/processed/warsaw_apartments.parquet')
print(df.shape)
df.head()

## Train / Test Split

In [ ]:
X = df.drop(columns=['price', 'pricePerSqm'])
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## Baseline Model Comparison

In [ ]:
baseline_models = {
    'LightGBM': lgb.LGBMRegressor(random_state=42, n_estimators=500, learning_rate=0.05),
    'XGBoost': xgb.XGBRegressor(random_state=42, n_estimators=500, learning_rate=0.05, tree_method='hist'),
    'RandomForest': RandomForestRegressor(random_state=42, n_estimators=200, n_jobs=-1),
    'Ridge': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', Ridge())
    ])
}

results = []
for name, model in baseline_models.items():
    with mlflow.start_run(run_name=f"baseline-{name.lower()}"):
        mlflow.set_tag("phase", "baseline")
        print(f"Training {name}...")
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        rmse = root_mean_squared_error(y_test, y_pred)
        mape = mean_absolute_percentage_error(y_test, y_pred)
        mlflow.log_metrics({"rmse": rmse, "mape": round(mape * 100, 2)})
        results.append({'Model': name, 'RMSE': rmse, 'MAPE (%)': round(mape * 100, 2), 'Tuned': False})
        print(f"  RMSE: {rmse:,.0f} PLN | MAPE: {mape*100:.2f}%")

pd.DataFrame(results).sort_values('RMSE').reset_index(drop=True)

## Feature Importance (Random Forest baseline)

In [ ]:
importance = pd.Series(
    baseline_models['RandomForest'].feature_importances_,
    index=X.columns
).sort_values(ascending=True)

plt.figure(figsize=(10, 8))
importance.plot(kind='barh')
plt.title('Feature Importance \u2014 RandomForest')
plt.xlabel('Importance')
plt.axvline(x=0.005, color='red', linestyle='--', label='0.5% threshold')
plt.legend()
plt.tight_layout()
plt.show()

print("\nFeatures below 0.5% importance (candidates to drop):")
print(importance[importance < 0.005].sort_values().index.tolist())

## Drop Low-Importance Features

In [ ]:
low_importance = importance[importance < 0.005].index.tolist()
print(f"Dropping: {low_importance}")

X_train = X_train.drop(columns=low_importance)
X_test = X_test.drop(columns=low_importance)
print(f"Features remaining: {X_train.shape[1]}")

## Optuna Tuning \u2014 LightGBM

In [ ]:
def lgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'random_state': 42
    }
    model = lgb.LGBMRegressor(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return root_mean_squared_error(y_test, y_pred)

lgb_study = optuna.create_study(direction='minimize')
lgb_study.optimize(lgb_objective, n_trials=50)

with mlflow.start_run(run_name="optuna-lgb-best"):
    mlflow.set_tag("phase", "tuning")
    mlflow.log_params(lgb_study.best_params)
    mlflow.log_metric("rmse", lgb_study.best_value)

print(f"Best RMSE: {lgb_study.best_value:,.0f} PLN")
print(f"Best params: {lgb_study.best_params}")

## Optuna Tuning \u2014 XGBoost

In [ ]:
def xgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'tree_method': 'hist',
        'random_state': 42
    }
    model = xgb.XGBRegressor(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return root_mean_squared_error(y_test, y_pred)

xgb_study = optuna.create_study(direction='minimize')
xgb_study.optimize(xgb_objective, n_trials=50)

with mlflow.start_run(run_name="optuna-xgb-best"):
    mlflow.set_tag("phase", "tuning")
    mlflow.log_params(xgb_study.best_params)
    mlflow.log_metric("rmse", xgb_study.best_value)

print(f"Best RMSE: {xgb_study.best_value:,.0f} PLN")
print(f"Best params: {xgb_study.best_params}")

## Final Comparison

In [ ]:
tuned_lgb = lgb.LGBMRegressor(**lgb_study.best_params, random_state=42, verbose=-1)
tuned_lgb.fit(X_train, y_train)
lgb_pred = tuned_lgb.predict(X_test)

tuned_xgb = xgb.XGBRegressor(**xgb_study.best_params, random_state=42)
tuned_xgb.fit(X_train, y_train)
xgb_pred = tuned_xgb.predict(X_test)

rf = RandomForestRegressor(random_state=42, n_estimators=200, n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

final_results = [
    {'Model': 'LightGBM (tuned)', 'RMSE': root_mean_squared_error(y_test, lgb_pred), 'MAPE (%)': round(mean_absolute_percentage_error(y_test, lgb_pred) * 100, 2)},
    {'Model': 'XGBoost (tuned)',  'RMSE': root_mean_squared_error(y_test, xgb_pred), 'MAPE (%)': round(mean_absolute_percentage_error(y_test, xgb_pred) * 100, 2)},
    {'Model': 'RandomForest (reduced features)', 'RMSE': root_mean_squared_error(y_test, rf_pred), 'MAPE (%)': round(mean_absolute_percentage_error(y_test, rf_pred) * 100, 2)},
]

with mlflow.start_run(run_name="final-lgb"):
    mlflow.set_tag("phase", "final")
    mlflow.log_params({**lgb_study.best_params, "random_state": 42})
    mlflow.log_metrics({"rmse": root_mean_squared_error(y_test, lgb_pred), "mape": round(mean_absolute_percentage_error(y_test, lgb_pred) * 100, 2)})
    mlflow.lightgbm.log_model(tuned_lgb, artifact_path="model")

with mlflow.start_run(run_name="final-xgb"):
    mlflow.set_tag("phase", "final")
    mlflow.log_params({**xgb_study.best_params, "random_state": 42})
    mlflow.log_metrics({"rmse": root_mean_squared_error(y_test, xgb_pred), "mape": round(mean_absolute_percentage_error(y_test, xgb_pred) * 100, 2)})
    mlflow.xgboost.log_model(tuned_xgb, artifact_path="model")

with mlflow.start_run(run_name="final-rf"):
    mlflow.set_tag("phase", "final")
    mlflow.log_params({"n_estimators": 200, "random_state": 42})
    mlflow.log_metrics({"rmse": root_mean_squared_error(y_test, rf_pred), "mape": round(mean_absolute_percentage_error(y_test, rf_pred) * 100, 2)})
    mlflow.sklearn.log_model(rf, artifact_path="model")

final_df = pd.DataFrame(final_results).sort_values('RMSE').reset_index(drop=True)
final_df

In [ ]:
import joblib
import json

joblib.dump(tuned_xgb, '../models/xgboost_model.joblib')

with open('../models/feature_names.json', 'w') as f:
    json.dump(list(X_train.columns), f)

print("Saved model to models/xgboost_model.joblib")
print(f"Features: {list(X_train.columns)}")